# 03 - Semaglutide treatment episodes and pregnancy exposure

**Runs in:** Truveta Studio notebook environment only.

**What this notebook does**

Turns the raw semaglutide dispensing records into person-level treatment
metrics: continuous treatment **episodes**, discontinuation, reinitiation,
accumulated persistence, and how much of the pregnancy was exposed.

**Episode rule.** Fills are chained into one episode while the gap between the
end of the previous days-supply and the next fill date is `<= GAP_DAYS` (60
days). A gap longer than that starts a new episode and counts as a
discontinuation; a later fill after such a gap counts as a reinitiation.

**Inputs** (from notebook 01):

* `medication_full.csv` - all semaglutide dispensing records
* `test_t3.csv` - exposed cohort (for delivery date and estimated LMP)

**Outputs:**

| File | Contents |
|---|---|
| `drug_dayssupply.csv` | per-person episode summary incl. median days supply |
| `drug_source_label.csv` | `med` vs `med_wo_t2d` label used by notebook 05 |
| `drugexporsure.csv` | per-person exposure metrics (note: filename misspelling is intentional - downstream notebooks read this exact name) |
| `revise_results/drugpersistence_p.csv` | Table: persistence metrics by source type |
| `revise_results/drugexporsure_p.csv` | Table: persistence metrics by in-pregnancy exposure |

**Run order:** 01 -> 02 -> **03** -> 04 -> 05.

## 1. Setup and configuration

In [ ]:
from truveta.study import Client, OutputMode, display_df

import re
import warnings

import numpy as np
import pandas as pd
import pyspark.pandas as ps

warnings.filterwarnings("ignore")

In [ ]:
# ============================================================================
# CONFIG
# ============================================================================

POPULATION_TITLE = "Delivery"

GAP_DAYS = 60          # gap longer than this starts a new treatment episode

# Which formulation implies which prescribing indication.
FORMULATION_INDICATION = {
    "Diabetes": ["Ozempic", "Rybelsus"],
    "Obesity":  ["Wegovy"],
}

# --- input files (written by notebook 01) --------------------------------
IN_MEDICATION = "/medication_full.csv"
IN_COHORT     = "/test_t3.csv"

# --- output files ---------------------------------------------------------
OUT_DAYSSUPPLY   = "/drug_dayssupply.csv"
OUT_SOURCE_LABEL = "/drug_source_label.csv"
OUT_EXPOSURE     = "/drugexporsure.csv"          # spelling kept for compatibility
OUT_TABLE_PERSIST  = "/revise_results/drugpersistence_p.csv"
OUT_TABLE_EXPOSURE = "/revise_results/drugexporsure_p.csv"

In [ ]:
client = Client(output_mode=OutputMode.PandasOnSpark)
# client = Client(output_mode=OutputMode.PySpark)

study = client.get_study()
population = study.get_population(title=POPULATION_TITLE)
snapshot = population.get_latest_snapshot()

output_path_local = study.get_output_path(fs=True)
population

## 2. Load inputs

In [ ]:
medication_full = pd.read_csv(output_path_local + IN_MEDICATION)
delivery_df = pd.read_csv(output_path_local + IN_COHORT)

print("dispensing records:", medication_full.shape)
print("exposed cohort    :", delivery_df.shape)

In [ ]:
# Restrict dispensing records to the analytic cohort and attach pregnancy dates.
medication_df = medication_full[medication_full["PersonId"].isin(delivery_df["PersonId"])].copy()
medication_df = pd.merge(
    delivery_df[["PersonId", "delivery_date", "estimated_LMP"]],
    medication_df, on="PersonId", how="left")

print("records in cohort:", medication_df.shape)

## 3. Build treatment episodes

In [ ]:
def extract_formulation(code):
    """Pull the brand name out of a medication code description."""
    match = re.search(r"\[(.*?)\]", code)
    if match:
        return match.group(1).strip()
    for brand in ("Rybelsus", "Ozempic", "Wegovy"):
        if brand in code:
            return brand
    return "Unknown"


medication_df = medication_df.rename(columns={"DispenseDateTime": "med_date"})
medication_df["med_date"] = pd.to_datetime(medication_df["med_date"])

# Fills after delivery are irrelevant to this analysis.
medication_df = medication_df[medication_df["med_date"] <= medication_df["delivery_date"]]
medication_df = medication_df.sort_values(["PersonId", "med_date"])

medication_df["Formulation"] = medication_df["Code"].apply(extract_formulation)

In [ ]:
# One row per person per fill date: sum days supply, keep every code/brand seen.
medication_df = medication_df.groupby(["PersonId", "med_date"], as_index=False).agg({
    "DaysSupply": "sum",
    "Code": lambda x: ", ".join(sorted(set(x))),
    "delivery_date": "first",
    "estimated_LMP": "first",
    "Formulation": lambda x: ", ".join(sorted(set(x))),
})

# Coverage end date implied by the days supply on this fill.
medication_df["EndDate"] = medication_df["med_date"] + pd.to_timedelta(
    medication_df["DaysSupply"], unit="d")
medication_df.head()

In [ ]:
def assign_episodes(medication_df, gap_days=GAP_DAYS):
    """Chain fills into treatment episodes.

    A new episode starts whenever the next fill occurs more than `gap_days`
    after the previous fill's coverage end date.
    """
    records = []

    for _, group in medication_df.sort_values(["PersonId", "med_date"]).groupby("PersonId"):
        group = group.reset_index(drop=True)
        episode_id = 1
        prev_end = group.loc[0, "EndDate"]

        group.loc[0, "EpisodeID"] = episode_id
        records.append(group.loc[0])

        for i in range(1, len(group)):
            gap = (group.loc[i, "med_date"] - prev_end).days
            if gap > gap_days:
                episode_id += 1
            group.loc[i, "EpisodeID"] = episode_id
            prev_end = group.loc[i, "EndDate"]
            records.append(group.loc[i])

    return pd.DataFrame(records)


episode_annotated_df = assign_episodes(medication_df)
print(episode_annotated_df.DaysSupply.describe())
episode_annotated_df.head()

In [ ]:
# Discontinuation = gap to the next fill exceeds GAP_DAYS.
# Reinitiation    = such a gap that is nevertheless followed by another fill.
episode_df = episode_annotated_df.sort_values(["PersonId", "med_date"]).copy()

episode_df["NextStart"] = episode_df.groupby("PersonId")["med_date"].shift(-1)
episode_df["DiscontinuationTime"] = (episode_df["NextStart"] - episode_df["EndDate"]).dt.days
episode_df["IsReinitiation"] = (
    (episode_df["DiscontinuationTime"] > GAP_DAYS) & episode_df["NextStart"].notna())

episode_df.head()

## 4. Person-level treatment summary

In [ ]:
summary = episode_df.groupby("PersonId").agg(
    DrugStart=("med_date", "min"),
    DrugEnd=("EndDate", "max"),
    Supply=("DaysSupply", "median"),
    NumDrugEpisodes=("EpisodeID", lambda x: len(set(x))),
    Discontinued60=("DiscontinuationTime", lambda x: any(x > GAP_DAYS)),
    TotalDiscontinuationTime=("DiscontinuationTime", lambda x: x[x > GAP_DAYS].sum()),
    Reinitiation=("IsReinitiation", "any"),
).reset_index()

summary.to_csv(output_path_local + OUT_DAYSSUPPLY, index=False)
print("persons:", len(summary))
summary.Supply.describe()

## 5. Accumulated persistence

Two versions of "how long was this person actually on treatment":

* **before delivery** - episodes truncated at the delivery date
* **before pregnancy** - episodes truncated at the estimated LMP

In [ ]:
def accumulated_persistence(episode_df, anchor_col, out_col):
    """Sum episode durations, truncating each episode at `anchor_col`.

    Episodes that start after the anchor date are dropped entirely.
    """
    ep = episode_df.groupby(["PersonId", "EpisodeID"]).agg(
        EpisodeStart=("med_date", "min"),
        EpisodeEnd=("EndDate", "max"),
        anchor=(anchor_col, "first"),
    ).reset_index()

    ep["anchor"] = pd.to_datetime(ep["anchor"])
    ep["EpisodeEnd_Truncated"] = ep[["EpisodeEnd", "anchor"]].min(axis=1)
    ep = ep[ep["EpisodeStart"] <= ep["anchor"]]

    ep["duration"] = (ep["EpisodeEnd_Truncated"] - ep["EpisodeStart"]).dt.days

    return ep, ep.groupby("PersonId").agg(**{out_col: ("duration", "sum")}).reset_index()


delivery_episode_summary, accumulated_delivery = accumulated_persistence(
    episode_df, "delivery_date", "AccumulatedPersistenceBeforeDelivery")

preg_episode_summary, accumulated_preg_persistence = accumulated_persistence(
    episode_df, "estimated_LMP", "AccumulatedPersistenceBeforePregnancy")

print("episodes before delivery :", len(delivery_episode_summary))
print("episodes before pregnancy:", len(preg_episode_summary),
      "| persons:", preg_episode_summary.PersonId.nunique())
accumulated_preg_persistence.head()

## 6. Exposure during pregnancy

An episode contributes exposure if its coverage extends past the estimated LMP.
Exposure days are counted from the later of (episode start, LMP) to the episode
end.

In [ ]:
def pregnancy_exposure(row):
    """(was_exposed, exposure_days) for one fill's coverage window."""
    start, end, preg_start = row["med_date"], row["EndDate"], row["estimated_LMP"]

    if end <= preg_start:
        return (False, 0)

    exposure_days = (end - start).days if start >= preg_start else (end - preg_start).days
    return (True, exposure_days)


check_preg = episode_annotated_df.copy()
for c in ["estimated_LMP", "EndDate", "med_date"]:
    check_preg[c] = pd.to_datetime(check_preg[c])

check_preg[["WasExposedInPregnancy", "PregnancyExposureDays"]] = check_preg.apply(
    pregnancy_exposure, axis=1, result_type="expand")

pregnancy_df = check_preg.groupby("PersonId").agg(
    ExposedInPregnancy=("WasExposedInPregnancy", "any"),
    TotalExposureDaysInPregnancy=("PregnancyExposureDays", "sum"),
).reset_index()

pregnancy_df.head()

## 7. Prescribing indication and the `med_wo_t2d` label

`med_indict` is inferred from the brand dispensed (Ozempic/Rybelsus -> diabetes,
Wegovy -> obesity).

`source_type` is then refined: a medication user is relabelled `med_wo_t2d`
("semaglutide for weight, not diabetes") if they have **no pre-pregnancy T2D**,
or if they have T2D but their first dispense was an obesity formulation.

In [ ]:
# Reference label for whether the person had T2D before pregnancy.
delivery_df.loc[(delivery_df["t2d_before_pregnancy"] == False) &
                (delivery_df["source_type"] == "med"), "source_type"] = "med_wo_t2d"
print(delivery_df.source_type.value_counts())

In [ ]:
conditions = [
    episode_df["Formulation"].isin(FORMULATION_INDICATION["Diabetes"]),
    episode_df["Formulation"].isin(FORMULATION_INDICATION["Obesity"]),
]
choices = ["Diabetes", "Obesity"]

episode_df1 = episode_df.copy()
episode_df1["med_indict"] = np.select(conditions, choices, default="Unknown")
episode_df1 = pd.merge(
    episode_df1,
    delivery_df[["PersonId", "t2d_before_pregnancy", "source_type"]],
    on="PersonId", how="left")

print("episode rows:", len(episode_df1))

In [ ]:
# Use the FIRST fill per person to decide the indication label.
episode_df1["med_date"] = pd.to_datetime(episode_df1["med_date"])
episode_df1 = episode_df1.sort_values(["PersonId", "med_date"])
episode_df2 = episode_df1.drop_duplicates(subset="PersonId", keep="first")

no_diabetes = episode_df2["t2d_before_pregnancy"] == False
has_diabetes_but_obesity = ((episode_df2["t2d_before_pregnancy"] == True) &
                            (episode_df2["med_indict"] == "Obesity"))

episode_df2.loc[no_diabetes | has_diabetes_but_obesity, "source_type"] = "med_wo_t2d"

drug_source_label = pd.merge(summary[["PersonId"]],
                             episode_df2[["PersonId", "source_type"]],
                             on="PersonId", how="left")
drug_source_label.to_csv(output_path_local + OUT_SOURCE_LABEL, index=False)
drug_source_label["source_type"].value_counts()

## 8. Assemble and export the exposure file

> People who **started** semaglutide during pregnancy are absent from
> `AccumulatedPersistenceBeforePregnancy` (they have no pre-pregnancy episode).
> `summary_df` uses inner joins and therefore excludes them; `drug_df` below
> uses a left join and fills their persistence with 0. `drug_df` is what gets
> exported, and notebook 05 uses the difference between the two to identify
> in-pregnancy initiators.

In [ ]:
# Inner-joined version: only people with pre-pregnancy treatment.
summary_df = (summary
              .merge(accumulated_delivery, on="PersonId")
              .merge(accumulated_preg_persistence, on="PersonId")
              .merge(pregnancy_df, on="PersonId"))

print("persons:", len(summary_df),
      "| with zero pre-pregnancy persistence:",
      len(summary_df[summary_df["AccumulatedPersistenceBeforePregnancy"] == 0]))

In [ ]:
# Exported version: everyone, with missing pre-pregnancy persistence set to 0.
drug_df = (summary
           .merge(accumulated_delivery, on="PersonId")
           .merge(accumulated_preg_persistence, on="PersonId", how="left")
           .merge(pregnancy_df, on="PersonId"))

drug_df["AccumulatedPersistenceBeforePregnancy"] = (
    drug_df["AccumulatedPersistenceBeforePregnancy"].fillna(0))
drug_df = drug_df.merge(episode_df2[["PersonId", "med_indict", "source_type"]], on="PersonId")

print("persons:", len(drug_df))
print(drug_df.isna().sum())

drug_df.to_csv(output_path_local + OUT_EXPOSURE, index=False)

## 9. Descriptive tables

In [ ]:
!pip install tableone
from tableone import TableOne

In [ ]:
PERSISTENCE_COLUMNS = [
    "NumDrugEpisodes", "TotalDiscontinuationTime", "Reinitiation",
    "AccumulatedPersistenceBeforeDelivery", "AccumulatedPersistenceBeforePregnancy",
    "ExposedInPregnancy", "TotalExposureDaysInPregnancy", "med_indict",
]
PERSISTENCE_CATEGORICAL = ["Reinitiation", "ExposedInPregnancy", "med_indict"]

In [ ]:
# By source type (med vs med_wo_t2d).
summary_df_labelled = (summary_df
                       .merge(drug_source_label, on="PersonId")
                       .merge(episode_df2[["PersonId", "med_indict"]], on="PersonId"))
print("persons:", len(summary_df_labelled))

table1 = TableOne(summary_df_labelled, columns=PERSISTENCE_COLUMNS,
                  categorical=PERSISTENCE_CATEGORICAL, groupby="source_type", pval=True)
table1.to_csv(output_path_local + OUT_TABLE_PERSIST, index=True)
table1

In [ ]:
# By whether treatment overlapped the pregnancy.
table1_exposure = TableOne(drug_df, columns=PERSISTENCE_COLUMNS,
                           categorical=PERSISTENCE_CATEGORICAL,
                           groupby="ExposedInPregnancy", pval=True)
table1_exposure.to_csv(output_path_local + OUT_TABLE_EXPOSURE, index=True)
table1_exposure

---

## Appendix - not used in the analysis

### A1. Formulation switching

Detects Ozempic <-> Wegovy <-> Rybelsus switches between consecutive fills.
Was explored but not reported.

In [ ]:
# --- NOT USED - DO NOT RUN ---
# episode_df = episode_df.sort_values(["PersonId", "med_date"])
# episode_df["PrevFormulation"] = episode_df.groupby("PersonId")["Formulation"].shift(1)
#
# def detect_change(row):
#     if pd.isna(row["PrevFormulation"]) or row["Formulation"] == row["PrevFormulation"]:
#         return None
#     return f"{row['PrevFormulation']} -> {row['Formulation']}"
#
# episode_df["DrugChange"] = episode_df.apply(detect_change, axis=1)
# episode_df.DrugChange.value_counts()